# Research 49 — One-hot cross-seed average

Average the aligned fold-safe one-hot VIP × region OOF probabilities from CV seeds 42 and 2025 with a fixed 50/50 mean, then use the predeclared fixed 80/20 reference mixture. Compare with each single-seed blend and the current four-seed leader; report macro/weighted F1 and fold-slice deltas on both partitions. No fitting, weight search, or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    sums = probs.sum(axis=1, keepdims=True)
    assert (sums > 0).all(), filename
    return probs / sums


p_base = load_probs("segment-oof.csv")
p_leader = load_probs("segment-vip-region-four-seed-average-oof.csv")
p_seed42 = load_probs("segment-lightgbm-onehot-vip-region-oof.csv")
p_seed2025 = load_probs("segment-lightgbm-onehot-vip-region-seed-2025-oof.csv")
p_blend42 = 0.8 * p_base + 0.2 * p_seed42
p_blend2025 = 0.8 * p_base + 0.2 * p_seed2025
p_mean = 0.5 * p_seed42 + 0.5 * p_seed2025
p_mean_blend = 0.8 * p_base + 0.2 * p_mean
folds = {
    seed: list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
    )
    for seed in [42, 2025]
}
candidates = {
    "Current reference": (p_base, None),
    "Four-seed VIP × region leader": (p_leader, None),
    "One-hot seed 42, fixed 80/20": (p_blend42, 42),
    "One-hot seed 2025, fixed 80/20": (p_blend2025, 2025),
    "One-hot two-seed mean, standalone": (p_mean, None),
    "One-hot two-seed mean, fixed 80/20": (p_mean_blend, "both"),
}
rows = []
for name, (probs, fold_seed) in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    seeds = [42, 2025] if fold_seed == "both" else ([fold_seed] if fold_seed is not None else [])
    for seed in seeds:
        for fold, (_, valid_idx) in enumerate(folds[seed], start=1):
            fold_score = f1_score(y[valid_idx], pred[valid_idx], average="macro")
            baseline_score = f1_score(
                y[valid_idx], p_base[valid_idx].argmax(axis=1), average="macro"
            )
            row[f"seed{seed}_fold{fold}_macro_f1"] = fold_score
            row[f"seed{seed}_fold{fold}_delta_vs_reference"] = fold_score - baseline_score
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-onehot-crossseed-blend.csv", index=False)
for probs, filename in [
    (p_mean, "segment-lightgbm-onehot-vip-region-two-seed-average-oof.csv"),
    (p_mean_blend, "segment-lightgbm-onehot-vip-region-two-seed-average-blend-oof.csv"),
]:
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / "predictions" / filename, index=False
    )
print("Fixed 50/50 seed mean and 80/20 blend evaluated; no weights searched; Test not loaded.")

,candidate,macro_f1,weighted_f1,seed42_fold1_macro_f1,seed42_fold1_delta_vs_reference,seed42_fold2_macro_f1,seed42_fold2_delta_vs_reference,seed42_fold3_macro_f1,seed42_fold3_delta_vs_reference,seed2025_fold1_macro_f1,seed2025_fold1_delta_vs_reference,seed2025_fold2_macro_f1,seed2025_fold2_delta_vs_reference,seed2025_fold3_macro_f1,seed2025_fold3_delta_vs_reference
0,Current reference,0.750895,0.764683,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Four-seed VIP × region leader,0.752172,0.765704,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"One-hot seed 42, fixed 80/20",0.751402,0.764894,0.749956,0.000511,0.750326,0.000506,0.753895,0.000506,NaN,NaN,NaN,NaN,NaN,NaN
3,"One-hot seed 2025, fixed 80/20",0.751457,0.764999,NaN,NaN,NaN,NaN,NaN,NaN,0.751245,0.000448,0.750191,0.000779,0.752927,0.000465
4,"One-hot two-seed mean, standalone",0.745777,0.757316,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,"One-hot two-seed mean, fixed 80/20",0.751323,0.764876,0.749635,0.000190,0.750409,0.000588,0.753891,0.000501,0.751235,0.000437,0.750244,0.000833,0.752482,0.000020


Fixed 50/50 seed mean and 80/20 blend evaluated; no weights searched; Test not loaded.
